In [ ]:
# Install Unsloth + training stack
import subprocess, shutil, os, sys

def run(cmd):
    # Use the running Python's pip — avoids PATH confusion on Kaggle
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FAILED [{cmd[:60]}]:\n{result.stderr[-2000:]}")
    else:
        print(f"OK: {cmd[:80]}")

PIP = f"{sys.executable} -m pip"

# 1. Pin numpy FIRST — Kaggle ships 2.0.2; unsloth deps upgrade to 2.4.x which
run(f"{PIP} install -q 'numpy<2.1'")

# 2. Install unsloth from git — this will pull unsloth_zoo from PyPI as a dep (that's fine,
run(f"{PIP} install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'")

# 3. Override unsloth_zoo with the git version AFTER unsloth — the PyPI version is missing
run(f"{PIP} install -q --force-reinstall --no-deps 'unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git'")

# 4. Other deps
run(f"{PIP} install -q 'bitsandbytes>=0.43.0'")
run(f"{PIP} install -q tqdm")

# 5. Clear Unsloth's compiled kernel cache — stale entries from a previous
cache_dir = "/kaggle/working/unsloth_compiled_cache"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"Cleared stale compiled cache: {cache_dir}")

# 6. Verify the installed unsloth_zoo is the git version (should have today's date or
result = subprocess.run(f"{PIP} show unsloth_zoo", shell=True, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if line.startswith(("Name", "Version", "Location")):
        print(line)

print("\nDone. Restart the session now (Run → Restart Session), then run all cells from cell 2 onwards.")


In [ ]:
# Authenticate with HuggingFace

import os
from huggingface_hub import login

# Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception:
    # Colab secrets fallback
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded from Colab Secrets")
    except Exception:
        hf_token = os.environ.get("HF_TOKEN", "")
        print("HF_TOKEN loaded from environment variable")

if not hf_token:
    raise ValueError(
        "HF_TOKEN not found. Add it as a Kaggle Secret or set environment variable HF_TOKEN."
    )

login(token=hf_token, add_to_git_credential=False)
print("HuggingFace login successful")


In [ ]:
# Config — edit these values

HF_REPO_ID = "YOUR_HF_USERNAME/lumen-medical-8b"

# Kaggle dataset path — after uploading your JSONL files as a Kaggle Dataset
# Path format: /kaggle/input/DATASET-NAME/filename.jsonl
DAPT_DATA_PATH = "/kaggle/input/lumen-training-data/dapt_corpus.jsonl"
SFT_DATA_PATH  = "/kaggle/input/lumen-training-data/sft_pairs.jsonl"

# WandB project name (free at wandb.ai) — set to None to disable
WANDB_PROJECT = "lumen-finetune"

# Base model
BASE_MODEL = "aaditya/Llama3-OpenBioLLM-8B"

# Output directories (Kaggle working dir persists during session)
DAPT_OUTPUT_DIR = "/kaggle/working/lumen-dapt"
SFT_OUTPUT_DIR  = "/kaggle/working/lumen-sft"
MERGED_DIR      = "/kaggle/working/lumen-merged"

print("Config loaded:")
print(f"  Base model : {BASE_MODEL}")
print(f"  DAPT data  : {DAPT_DATA_PATH}")
print(f"  SFT data   : {SFT_DATA_PATH}")
print(f"  HF repo    : {HF_REPO_ID}")


In [ ]:
# GPU check + WandB init
import os

# Must be set BEFORE torch initializes CUDA — cannot be set inside a later cell
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"

import torch

# GPU check
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable GPU: Settings → Accelerator → GPU T4")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}  ({vram_gb:.1f} GB VRAM)")
print("WandB disabled")
print("PYTORCH_ALLOC_CONF=expandable_segments:True")


In [ ]:
# Phase 1: Load base model with Unsloth (4-bit QLoRA)
import torch
from unsloth import FastLanguageModel

torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name       = BASE_MODEL,
    max_seq_length   = 1024,     # Reduced from 2048 — attention is O(L²), halving L cuts KV cache 4x
    dtype            = None,
    load_in_4bit     = True,
    token            = hf_token,
)

used_gb = torch.cuda.memory_allocated() / 1e9
print(f"Model loaded. VRAM used: {used_gb:.1f} GB")

model = FastLanguageModel.get_peft_model(
    model,
    r                = 32,
    target_modules   = ["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
    lora_alpha       = 32,
    lora_dropout     = 0,        # 0 required for Unsloth fast kernels
    bias             = "none",
    use_gradient_checkpointing = "unsloth",
    random_state     = 42,
)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params / 1e6:.1f}M")
print(f"Trainable (LoRA): {trainable / 1e6:.1f}M  ({100*trainable/total_params:.2f}%)")


In [ ]:
# Phase 1: Load DAPT dataset
import json
from datasets import Dataset

def load_dapt_dataset(path: str) -> Dataset:
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                text = rec.get("text", "").strip()
                # Filter very short texts (< 50 chars) — not useful for training
                if len(text) >= 50:
                    records.append({"text": text})
            except json.JSONDecodeError:
                continue
    print(f"Loaded {len(records):,} DAPT records from {path}")
    return Dataset.from_list(records)

dapt_dataset = load_dapt_dataset(DAPT_DATA_PATH)

# Quick sanity check — show 2 samples
print("\nSample 0 (first 200 chars):")
print(dapt_dataset[0]["text"][:200])
print("\nSample -1 (last 200 chars):")
print(dapt_dataset[-1]["text"][:200])


In [ ]:
# Phase 1: DAPT Training
import os
from trl import SFTTrainer, SFTConfig

resume_checkpoint = None

if os.path.isdir(DAPT_OUTPUT_DIR):
    local_ckpts = sorted(
        [d for d in os.listdir(DAPT_OUTPUT_DIR) if d.startswith("checkpoint-")],
        key=lambda x: int(x.split("-")[1])
    )
    if local_ckpts:
        resume_checkpoint = os.path.join(DAPT_OUTPUT_DIR, local_ckpts[-1])
        print(f"Resuming from LOCAL checkpoint: {resume_checkpoint}")

if resume_checkpoint is None:
    try:
        from huggingface_hub import snapshot_download, list_repo_refs
        refs = list_repo_refs(HF_REPO_ID, token=hf_token)
        branch_names = [b.name for b in refs.branches]
        if "checkpoint" in branch_names:
            print("Found checkpoint branch on HF Hub — downloading...")
            local_hub_ckpt = os.path.join(DAPT_OUTPUT_DIR, "hf_checkpoint_resume")
            resume_checkpoint = snapshot_download(
                repo_id=HF_REPO_ID,
                revision="checkpoint",   # the branch hub_strategy="checkpoint" writes to
                token=hf_token,
                local_dir=local_hub_ckpt,
            )
            print(f"Resuming from HF Hub checkpoint: {resume_checkpoint}")
        else:
            print("No checkpoint branch on HF Hub — training from scratch.")
    except Exception as e:
        print(f"HF Hub check failed ({e}) — training from scratch.")

# Config
dapt_config = SFTConfig(
    output_dir                  = DAPT_OUTPUT_DIR,
    num_train_epochs            = 1,
    per_device_train_batch_size = 1,       # batch=1 → effective batch=16 → ~1024 steps over 41K records
    gradient_accumulation_steps = 16,
    warmup_steps                = 100,
    learning_rate               = 2e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    seed                        = 42,
    logging_steps               = 25,
    save_steps                  = 50,
    save_total_limit            = 3,
    report_to                   = "none",
    run_name                    = "lumen-dapt-phase1",
    dataset_text_field          = "text",
    max_seq_length              = 1024,
    packing                     = True,
    # Push every checkpoint to HF Hub so it survives /kaggle/working wipes
    push_to_hub                 = True,
    hub_model_id                = HF_REPO_ID,
    hub_strategy                = "checkpoint",
    hub_private_repo            = True,
)

dapt_trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dapt_dataset,
    args          = dapt_config,
)

total_steps = len(dapt_trainer.get_train_dataloader()) // dapt_config.gradient_accumulation_steps
print(f"\nStarting Phase 1 DAPT training...")
print(f"  Dataset size        : {len(dapt_dataset):,} records")
print(f"  Total steps         : {total_steps}")
print(f"  per_device_batch=1 × grad_accum=16 → effective batch=16")
print(f"  Checkpoints pushed to HF Hub every {dapt_config.save_steps} steps")
print(f"  Resume: {resume_checkpoint or 'scratch'}")
print()

dapt_trainer_stats = dapt_trainer.train(resume_from_checkpoint=resume_checkpoint)

print(f"\nPhase 1 complete.")
print(f"  Training loss : {dapt_trainer_stats.training_loss:.4f}")
print(f"  Runtime       : {dapt_trainer_stats.metrics['train_runtime'] / 60:.1f} minutes")


In [ ]:
# Phase 2: Reconfigure LoRA for SFT

import torch
from unsloth import FastLanguageModel

# Clear VRAM from Phase 1 trainer before Phase 2
del dapt_trainer
torch.cuda.empty_cache()

# Re-add LoRA with lower rank for SFT (schema learning, not deep domain adaptation)
model = FastLanguageModel.get_peft_model(
    model,
    r                = 16,         # Lower rank than DAPT — schema is simpler to learn
    target_modules   = ["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
    lora_alpha       = 16,
    lora_dropout     = 0,
    bias             = "none",
    use_gradient_checkpointing = "unsloth",
    random_state     = 42,
)

used_gb = torch.cuda.memory_allocated() / 1e9
print(f"Phase 2 LoRA attached. VRAM used: {used_gb:.1f} GB")


In [ ]:
# Phase 2: Load SFT dataset

import json
from datasets import Dataset

def load_sft_dataset(path: str) -> Dataset:
    records = []
    skipped = 0
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                convs = rec.get("conversations", [])
                # Must have system + user + assistant turns
                if len(convs) < 3:
                    skipped += 1
                    continue
                # Validate assistant turn is non-empty JSON
                assistant = next((c for c in convs if c.get("role") == "assistant"), None)
                if not assistant or not assistant.get("content", "").strip():
                    skipped += 1
                    continue
                records.append({"conversations": convs})
            except json.JSONDecodeError:
                skipped += 1
                continue

    print(f"Loaded  : {len(records):,} valid SFT pairs")
    print(f"Skipped : {skipped}")
    return Dataset.from_list(records)


sft_dataset = load_sft_dataset(SFT_DATA_PATH)

# Apply Llama-3 chat template — converts conversations list to a single formatted string
def format_chat_template(examples):
    texts = []
    for convs in examples["conversations"]:
        # tokenizer.apply_chat_template handles Llama-3 <|im_start|> tokens correctly
        text = tokenizer.apply_chat_template(
            convs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

sft_dataset = sft_dataset.map(format_chat_template, batched=True)

print("\nSample formatted prompt (first 400 chars):")
print(sft_dataset[0]["text"][:400])


In [ ]:
# Phase 2: SFT Training
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    # Paths
    output_dir                  = SFT_OUTPUT_DIR,
    # Training loop
    num_train_epochs            = 3,       # 3 epochs on 252 pairs — small dataset needs more passes
    per_device_train_batch_size = 1,       # SFT sequences are 3-4K tokens; keep batch at 1
    gradient_accumulation_steps = 8,       # Effective batch = 8
    warmup_steps                = 20,
    learning_rate               = 1e-4,    # Lower than DAPT — prevents catastrophic forgetting
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "cosine",
    seed                        = 42,
    # Logging / checkpoints
    logging_steps               = 10,
    save_steps                  = 50,
    save_total_limit            = 2,
    report_to                   = "none",
    run_name                    = "lumen-sft-phase2",
    # SFT-specific
    dataset_text_field          = "text",
    max_seq_length              = 4096,    # Lumen JSON outputs are 3–4K tokens
    packing                     = False,   # Do NOT pack SFT — each example must complete fully
)

sft_trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = sft_dataset,
    args          = sft_config,
)

print("Starting Phase 2 SFT training...")
print(f"  Dataset size        : {len(sft_dataset):,} pairs")
print(f"  Epochs              : 3")
print(f"  Effective batch size: {1 * 8}")
print(f"  Max seq length      : 4096 tokens")
print(f"  Checkpoints         → {SFT_OUTPUT_DIR}")
print()

sft_trainer_stats = sft_trainer.train()

print(f"\nPhase 2 complete.")
print(f"  Training loss : {sft_trainer_stats.training_loss:.4f}")
print(f"  Runtime       : {sft_trainer_stats.metrics['train_runtime'] / 60:.1f} minutes")


In [ ]:
# Quick inference test on fine-tuned model
import json
import torch
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)  # Switch to inference mode (2x faster)

TEST_SYSTEM_PROMPT = (
    "You are Lumen, a medical report explainer for Indian patients. "
    "Output ONLY valid JSON. No markdown, no commentary. "
    "Extract all test results and return them with explanations in simple Indian English."
)

TEST_REPORT = """
THYROCARE TECHNOLOGIES LIMITED
Patient: Ramesh Kumar, 45 years Male, Mumbai
Ref By: Dr. Suresh Mehta

TEST                     RESULT   UNIT        REFERENCE RANGE
------------------------------------------------------------------
Haemoglobin              10.2     g/dL        13.0-17.0
Total WBC Count          11800    cells/uL    4000-11000
Platelet Count           145000   cells/uL    150000-410000
MCV                      72       fL          80-100
MCH                      22       pg          27-32
MCHC                     30       g/dL        31.5-34.5
RBC Count                3.8      mill/cmm    4.5-5.5
Neutrophils              78       %           40-70
Lymphocytes              16       %           20-40
"""

messages = [
    {"role": "system", "content": TEST_SYSTEM_PROMPT},
    {"role": "user",   "content": (
        "Analyse the medical document below and return a JSON object.\n\n"
        f"Input data:\n{{\"parsed_data\": {{\"raw_text\": {json.dumps(TEST_REPORT)}, "
        f"\"tests\": [], \"medicines\": []}}}}\n\nReturn ONLY JSON."
    )},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids       = inputs,
        max_new_tokens  = 1500,
        temperature     = 0.1,
        do_sample       = True,
        pad_token_id    = tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("Model output (first 1500 chars)")
print(response[:1500])

# Validate JSON
print("\nJSON Validation")
try:
    parsed = json.loads(response)
    required = {"disclaimer", "abnormal_values", "normal_values", "medicines",
                "overall_summary", "urgency_level", "confidence_score"}
    missing = required - set(parsed.keys())
    if missing:
        print(f"WARNING: Missing keys: {missing}")
    else:
        print("PASS: All required top-level keys present")
        print(f"  abnormal_values : {len(parsed.get('abnormal_values',[]))} items")
        print(f"  normal_values   : {len(parsed.get('normal_values',[]))} items")
        print(f"  urgency_level   : {parsed.get('urgency_level')}")
        print(f"  confidence_score: {parsed.get('confidence_score')}")
except json.JSONDecodeError as e:
    print(f"FAIL: Output is not valid JSON — {e}")
    print("Action: Add more SFT pairs and re-run Phase 2 with higher epochs")


---
## Step 4 — Merge LoRA Adapters and Upload to HuggingFace

Merge the LoRA weights back into the base model weights to produce a standalone model
that can be loaded without the PEFT library (required for Ollama and vLLM).

**Only run this cell if the inference test above passed.**


In [ ]:
# Merge adapters + save merged model
import os, shutil, torch

# Switch back to training mode before merging
FastLanguageModel.for_training(model)

print("Merging LoRA adapters into base weights...")
# merge_and_unload fuses the adapter weights into the base model parameters
# Result: a standard HuggingFace model with no adapter dependency
merged_model = model.merge_and_unload()
print("Merge complete")

# Save merged model + tokenizer locally first
print(f"\nSaving merged model to {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

# Disk usage
total_bytes = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk(MERGED_DIR)
    for f in files
)
print(f"Merged model size: {total_bytes / 1e9:.2f} GB")
print(f"Files: {os.listdir(MERGED_DIR)}")


In [ ]:
# Push merged model to HuggingFace Hub
# Creates/updates a private repo at huggingface.co/YOUR_USERNAME/lumen-medical-8b
# Prerequisites:
#   1. Create the repo manually at huggingface.co/new (set Private)
#   2. HF token must have Write permissions

from huggingface_hub import HfApi
import os

print(f"Uploading to HuggingFace repo: {HF_REPO_ID}")
print("(This will take 5–15 minutes for a 16GB model — do not interrupt)\n")

# Upload the entire merged directory
api = HfApi(token=hf_token)
api.upload_folder(
    repo_id     = HF_REPO_ID,
    folder_path = MERGED_DIR,
    repo_type   = "model",
    commit_message = "Lumen fine-tuned: OpenBioLLM-8B + Indian Medical DAPT + Lumen SFT",
)

print(f"\nUpload complete!")
print(f"Model available at: https://huggingface.co/{HF_REPO_ID}")
print()
print("Next: Pull model locally with Ollama:")
print(f"  ollama pull hf.co/{HF_REPO_ID}")
print()
print("Or load in Lumen backend (set in .env):")
print("  LLM_PROVIDER=llama")
print(f"  LLAMA_MODEL=hf.co/{HF_REPO_ID}")


In [ ]:
# HuggingFace Inference API test (optional fallback)

from huggingface_hub import InferenceClient
import json

client = InferenceClient(
    model    = HF_REPO_ID,
    token    = hf_token,
)

test_messages = [
    {
        "role": "system",
        "content": "You are Lumen, a medical report explainer. Output ONLY valid JSON."
    },
    {
        "role": "user",
        "content": (
            "Analyse this CBC report:\n"
            "Haemoglobin: 10.2 g/dL (Normal: 13-17)\n"
            "WBC: 11800 cells/uL (Normal: 4000-11000)\n"
            "Return ONLY JSON with keys: disclaimer, abnormal_values, normal_values, overall_summary."
        )
    }
]

response = client.chat.completions.create(
    messages       = test_messages,   # must be list of dicts — NOT a string
    max_tokens     = 800,
    temperature    = 0.1,
)

output = response.choices[0].message.content
print("API response:")
print(output[:600])

try:
    json.loads(output)
    print("\nValid JSON output from Inference API")
except json.JSONDecodeError:
    print("\nOutput is not valid JSON — model may need warming up or more SFT data")
